In [7]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# *Housing Dataset*

In [8]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


# *Pre-processing*

In [9]:
# min_val and max_val are the boundaries for outliers
mean = torch.tensor(x_train.iloc[:, :6].mean().values)
std = torch.tensor(x_train.iloc[:, :6].std().values)
min_val = torch.tensor(x_train.iloc[:, :6].min().values)
max_val = torch.tensor(x_train.iloc[:, :6].max().values)

class PreprocessingLayer(nn.Module):
    def __init__(self, mean, std, min_val, max_val):
        super().__init__()
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)
        self.register_buffer('min_val', min_val)
        self.register_buffer('max_val', max_val)

    def forward(self, x):
        # Separating latitude and longitude
        x_process = x[:, :6]
        coords = x[:, 6:]
        # Outliers
        x_clipped = torch.clamp(x_process, self.min_val, self.max_val)
        # Standard Scaling - Manually
        x_scaled = (x_clipped - self.mean) / (self.std)
        # Concatenating back coords
        return torch.cat([x_scaled, coords], dim=1)

class HousingPred(nn.Module):
    def __init__(self, mean, std, min_val, max_val, hidden_size):
        super().__init__()
        self.prepro = PreprocessingLayer(mean, std, min_val, max_val)

        self.Model = nn.Sequential(
            nn.Linear(8, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x):
        x_ready = self.prepro(x)
        return self.Model(x_ready)

# *Training and evaluation functions*

In [10]:
def train_model(model, train_loader, val_loader, epochs, learning_rate, device):

    optimizer = optim.Adam(model.parameters(), lr = learning_rate)
    loss_func = nn.MSELoss()

    train_losses = []
    val_losses = []

    model.to(device)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero.grad()
            predictions = model(x_batch)
            loss = loss_func(predictions, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                predictions = model(X_batch)
                loss = loss_func(predictions, y_batch)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    return train_losses, val_losses

def evaluate_model (model, test_loader, device):
    model.eval()
    loss_func = nn.MSELoss()
    total_loss = 0.0
    predictions = []
    targets = []

    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            pred = model(x_batch)
            loss = loss_func(pred, y_batch)
            total_loss += loss.item()
            predictions.append(pred.cpu())
            targets.append(y_batch.cpu())

    mse = total_loss / len(test_loader)
    rmse = torch.sqrt(torch.tensor(mse))

    predictions = torch.cat(predictions)
    targets = torch.cat(targets)
    mae = torch.mean(torch.abs(predictions - targets))

    return mse, rmse, mae, predictions, targets




# *Training*